# 21 · Classify into a large taxonomy, level by level

## The problem

Real product catalogs, support-ticket taxonomies and document trees have **hundreds or thousands of leaves**. A shop
might have *Electronics › Phones › Chargers & cables* and 900 siblings like it. Two naive approaches both hurt:

- **One giant list of every leaf.** The options blur together, the prompt gets huge, and near-duplicates
  (*Men › Shoes* vs *Women › Shoes*) are easy to confuse.
- **Keyword rules.** They break the moment a title says "braided USB-C cord" instead of "charging cable".

And there's a third trap: items that **don't belong anywhere** in your taxonomy (a bag of dog food in an electronics,
home and clothing store). A classifier that is forced to pick will confidently file it somewhere wrong.

## Why Jev

Walking a tree means **one small decision per level**, each over just a handful of children. That's exactly a
System One job: a typed `Choice` with a **calibrated probability for every option**, in about 100 ms, for a
fraction of a cent. Because we get the full distribution (not just a label), we can:

- add a `none_of_these` option at every level, so "doesn't fit" is a first-class answer,
- notice when the top two branches are **close** and keep both (a *beam*) instead of committing too early,
- multiply the step probabilities into a **path probability** you can threshold or show to a human.

## What we'll build

```
product title ──► level 1 Choice ──► level 2 Choice ──► level 3 Choice ──► ranked paths
                  (3 depts + none)   (children + none)  (leaves + none)     e.g. Electronics › Phones › Chargers  0.97
                        │
                        └─ top two within CLOSE?  keep BOTH branches (beam) and walk each one
```

## What you'll learn
- How to turn a nested taxonomy into a sequence of small `Choice` questions, and why option **descriptions** matter.
- How a **beam** keeps the runner-up alive when the model is torn, and how path probabilities combine.
- How `none_of_these` lets the classifier say "this item is outside the taxonomy" instead of guessing.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #15.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads `TYPESAFE_API_KEY` from the `.env` file at the repo root (it searches this folder and its parents),
- creates the Jev client `jev`,
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so we can *see* distributions.

We import all the Jev question types for consistency with the other notebooks, but this one only needs **`Choice`**.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The taxonomy

A plain nested dict: departments → categories → a **list of leaves**. Real taxonomies are deeper and wider, but the
code below doesn't care about the size; it only ever looks at one node's children at a time.

Two small helpers:
- `children(node)` returns the option names at a node (dict keys, or the items of a leaf list).
- `describe(name, node)` builds the **description** Jev sees for an option. For a branch it lists the leaves
  underneath it (*"Furniture, which covers: Chairs, Desks, Shelving"*), so Jev knows what lives inside a branch
  before going down into it. A leaf is described by its own name.

This cell only **defines** data and helpers. Nothing is sent to Jev yet.

In [2]:
TAXONOMY = {
    "Electronics": {
        "Computers": ["Laptops", "Desktops", "Monitors", "Keyboards & mice"],
        "Phones": ["Smartphones", "Cases", "Chargers & cables"],
        "Audio": ["Headphones", "Speakers", "Microphones"],
    },
    "Home & Kitchen": {
        "Cookware": ["Pans & skillets", "Pots", "Knives"],
        "Appliances": ["Coffee makers", "Blenders", "Air fryers"],
        "Furniture": ["Chairs", "Desks", "Shelving"],
    },
    "Clothing": {
        "Men": ["Shirts", "Trousers", "Shoes"],
        "Women": ["Dresses", "Tops", "Shoes"],
        "Kids": ["Baby", "Boys", "Girls"],
    },
}
NONE = "none_of_these"


def children(node):
    """Option names at this node: the keys of a dict, or the items of a leaf list."""
    return list(node)


def leaves(node):
    """Every leaf name under a node (used to describe a branch)."""
    if isinstance(node, dict):
        return [leaf for sub in node.values() for leaf in leaves(sub)]
    return list(node)


def describe(name, node):
    """What Jev reads for one option: a branch lists its leaves, a leaf is just its name."""
    if isinstance(node, dict):
        return f"{name}, which covers: {', '.join(dict.fromkeys(leaves(node[name])))}"
    return name


for dept in children(TAXONOMY):
    print(describe(dept, TAXONOMY))

Electronics, which covers: Laptops, Desktops, Monitors, Keyboards & mice, Smartphones, Cases, Chargers & cables, Headphones, Speakers, Microphones
Home & Kitchen, which covers: Pans & skillets, Pots, Knives, Coffee makers, Blenders, Air fryers, Chairs, Desks, Shelving
Clothing, which covers: Shirts, Trousers, Shoes, Dresses, Tops, Baby, Boys, Girls


## Step 2 · One level = one `Choice`

**How a `Choice` works.** You give Jev `instructions` (the question) and `criteria`, a dict of
`{option_key: description}`. The answer has three parts:

| Field | Meaning |
|---|---|
| `.choice` | the winning option key |
| `.confidence` | how sure Jev is about that winner |
| `.probabilities` | a dict with a probability for **every** option (they sum to about 1) |

At each level we build a fresh `Choice` over that node's children, plus one extra option, `none_of_these`. `choose()`
returns the options **ranked by probability**, because the beam in Step 3 needs the runner-up, not just the winner.

The `state` (what Jev reads) is the product title. The `where` text tells Jev how deep we are
("Under Home & Kitchen > Furniture, which category...").

> **Workshop note:** the first version of this notebook described each option by its bare name (`"Home & Kitchen"`).
> With only that to go on, Jev decided an *ergonomic office chair* is **not** "Home & Kitchen" and answered
> `none_of_these`, which is a fair reading of those three words. Describing a branch by what's underneath it
> (`describe()` above) fixed it. Wording is the model's whole world: Jev only knows what your criteria say.

This cell only **defines** `choose()`.

In [3]:
def choose(item, options, path, client, node):
    """Ask Jev which child of `node` the item belongs to. Returns [(option, probability), ...] best first."""
    criteria = {o: describe(o, node) for o in options}
    criteria[NONE] = "None of these: the product does not belong under any of the categories listed"
    where = " > ".join(path) or "the top level"
    r = client.system_one(state=item, questions={"c": Choice(
        instructions=f"Under {where}, which category does this product belong to?", criteria=criteria)})
    return sorted(r.choices["c"].probabilities.items(), key=lambda kv: -kv[1])

### Try one level

Let's ask only the **top-level** question for three products and print the whole distribution, not just the winner.

In [4]:
for item in ["65W USB-C fast charger with braided cable",
             "Ergonomic mesh office chair with lumbar support",
             "20 lb bag of grain-free dog food"]:
    print(item)
    for option, p in choose(item, children(TAXONOMY), [], jev, TAXONOMY):
        print(f"   {option:<16} {bar(p)} {p:.2f}")

65W USB-C fast charger with braided cable


   Electronics      ████████████████████ 1.00
   Home & Kitchen   ···················· 0.00
   Clothing         ···················· 0.00
   none_of_these    ···················· 0.00
Ergonomic mesh office chair with lumbar support
   Home & Kitchen   ████████████████████ 0.98
   none_of_these    ···················· 0.02
   Electronics      ···················· 0.00
   Clothing         ···················· 0.00
20 lb bag of grain-free dog food


   none_of_these    ████████████████████ 1.00
   Electronics      ···················· 0.00
   Home & Kitchen   ···················· 0.00
   Clothing         ···················· 0.00


**What to notice**
- The charger goes to *Electronics* with nearly all the probability mass: an easy call.
- The office chair now lands in *Home & Kitchen*, because the description mentions Chairs, Desks and Shelving.
- The dog food puts almost everything on `none_of_these`. Our store doesn't sell pet supplies, and Jev says so instead
  of forcing it into the nearest department.
- Every option gets a probability. That full distribution is what makes the beam possible.

## Step 3 · Walk the tree with a beam

`classify()` is the main function. At each node it calls `choose()` and then decides which branches to follow:

| Situation at a node | What `classify()` does |
|---|---|
| top option is `none_of_these` | stop here; the path ends in `none_of_these` |
| runner-up is within `CLOSE` of the winner (and isn't `none`) | **beam:** walk **both** branches |
| otherwise | walk only the winner |
| the chosen option is a leaf | the path is complete |

**Path probability** = the product of the step probabilities along the path. If Jev gives *Electronics* 0.5, then
*Audio* 0.8, then *Speakers* 0.9, the path scores 0.5 × 0.8 × 0.9 = 0.36. That way two competing paths can be
compared fairly at the end, and a branch that looked slightly better at level 1 can still **lose** if it gets
shaky further down.

`CLOSE = 0.25` means "if the top two are within 25 points of each other, don't commit yet". Lower it to beam less
(cheaper, riskier); raise it to beam more (one extra Jev call per level per kept branch).

In [5]:
CLOSE = 0.25   # keep the runner-up when p(top) - p(second) is below this


def classify(item, client, node=None, path=(), p=1.0):
    """Return every explored path as [(path_list, path_probability), ...], best first."""
    if "TAXONOMY" not in globals():
        raise RuntimeError("TAXONOMY is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")
    node = TAXONOMY if node is None else node

    ranked = choose(item, children(node), list(path), client, node)
    (top, p1), (second, p2) = ranked[0], ranked[1]
    if top == NONE:                                         # out of taxonomy at this level
        return [(list(path) + [NONE], p * p1)]
    keep = [(top, p1)] + ([(second, p2)] if second != NONE and p1 - p2 < CLOSE else [])

    results = []
    for name, pk in keep:
        sub = node[name] if isinstance(node, dict) else None
        if sub is None:                                     # reached a leaf
            results.append((list(path) + [name], p * pk))
        else:
            results += classify(item, client, sub, path + (name,), p * pk)
    return sorted(results, key=lambda x: -x[1])

### Watch the beam fire

A **gaming chair with built-in Bluetooth speakers** is genuinely ambiguous: is it furniture or audio gear? First the
top-level distribution, then the full walk. If the beam fires, you'll see **more than one path** come back.

In [6]:
BEAM_ITEM = "Gaming chair with built-in Bluetooth speakers"

ranked = choose(BEAM_ITEM, children(TAXONOMY), [], jev, TAXONOMY)
for option, p in ranked:
    print(f"   {option:<16} {bar(p)} {p:.2f}")
gap = ranked[0][1] - ranked[1][1]
print(f"\ngap between top two = {gap:.2f}  ->  {'BEAM: keep both' if gap < CLOSE else 'commit to the winner'}\n")

for path, p in classify(BEAM_ITEM, jev):
    print(f"   {p:.2f}  {' > '.join(path)}")

   Electronics      ██████████·········· 0.49
   Home & Kitchen   █████████··········· 0.45
   none_of_these    █··················· 0.06
   Clothing         ···················· 0.00

gap between top two = 0.04  ->  BEAM: keep both



   0.40  Home & Kitchen > Furniture > Chairs
   0.31  Electronics > Audio > Speakers


**What to notice**
- At the top level *Electronics* and *Home & Kitchen* are close, well inside `CLOSE`, so the beam keeps **both**.
- Two complete paths come back, one under each department, each with its own path probability.
- The final ranking comes from the **whole path**, so the order can differ from the level-1 order: a branch that
  narrowly won level 1 can lose if its later steps are less certain. That's the point of not committing early.
- In a real app, two close paths are a good reason to show both to a human, or to list the item in both places.

## Step 4 · Classify a batch, including one that fits nowhere

Six ordinary products, the ambiguous gaming chair, and the dog food. For each one we print the best path and, when
the beam kept extra paths, the alternatives too.

In [7]:
ITEMS = [
    "65W USB-C fast charger with braided cable",
    "Noise-cancelling over-ear Bluetooth headphones",
    "10-inch pre-seasoned cast iron skillet",
    "Ergonomic mesh office chair with lumbar support",
    "Girls' rainbow light-up sneakers",
    BEAM_ITEM,
    "20 lb bag of grain-free dog food",
]
for item in ITEMS:
    paths = classify(item, jev)
    best, p = paths[0]
    print(f"{item[:46]:<46} -> {' > '.join(best)}  {p:.2f}")
    for alt, pa in paths[1:]:
        print(f"{'':<46}    also: {' > '.join(alt)}  {pa:.2f}")

65W USB-C fast charger with braided cable      -> Electronics > Phones > Chargers & cables  1.00


Noise-cancelling over-ear Bluetooth headphones -> Electronics > Audio > Headphones  1.00


10-inch pre-seasoned cast iron skillet         -> Home & Kitchen > Cookware > Pans & skillets  1.00


Ergonomic mesh office chair with lumbar suppor -> Home & Kitchen > Furniture > Chairs  0.94


Girls' rainbow light-up sneakers               -> Clothing > Kids > Girls  0.99


Gaming chair with built-in Bluetooth speakers  -> Home & Kitchen > Furniture > Chairs  0.44
                                                  also: Electronics > Audio > Speakers  0.29
20 lb bag of grain-free dog food               -> none_of_these  1.00


**What to notice**
- The clear-cut products each get **one** path with a high probability, three Jev calls each (one per level).
- The gaming chair is the only item that comes back with an `also:` line: the beam only costs extra calls when Jev is
  actually torn.
- The dog food ends on `none_of_these` at the **top level** after a single call, so out-of-taxonomy items are cheap to reject.
- `none_of_these` can also appear deeper down (e.g. `Clothing > none_of_these`), which means "right department, but no
  matching category". Try a unisex t-shirt in Step 6.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `classify()` with `ipywidgets`,
and an HTML card per product showing every path as breadcrumbs with its probability. The classification logic above is
untouched.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`taxonomy_html()` renders one card per product: the best path in green, beam alternatives in grey, and
`none_of_these` paths in slate. We render three examples: a clear one, the beam case, and the out-of-taxonomy one.

In [9]:
def taxonomy_html(item):
    paths = classify(item, jev)
    rows = ""
    for i, (path, p) in enumerate(paths):
        crumbs = " › ".join(f'<b>{s}</b>' if j == len(path) - 1 else s for j, s in enumerate(path))
        col = SLATE if NONE in path else SAGE if i == 0 else "#B9B4A8"
        rows += f'<div style="padding:4px 0;">{badge(f"{p:.2f}", col)} {crumbs}</div>'
    return box(f'<div style="{CARD}"><div style="color:{SLATE};font-size:13px;margin-bottom:6px;">{item}</div>{rows}</div>')


display(HTML(box("".join(taxonomy_html(i) for i in [ITEMS[1], BEAM_ITEM, ITEMS[-1]]))))

## Step 6 · Try it yourself

Edit the product title and press **Classify** (needs a live kernel). Good ones to try:
- `Stainless steel 8-cup pour-over coffee maker` → *Home & Kitchen › Appliances › Coffee makers*, one confident path.
- `Unisex cotton crew-neck t-shirt` → *Clothing*, then most likely `none_of_these`: the taxonomy only has Men / Women /
  Kids, and Jev won't pretend a unisex shirt is clearly one of them.
- `Keyboard tray that mounts under a desk` → torn between furniture and computer accessories; watch for a second path.
- `Yoga mat with carrying strap` → `none_of_these` at the top level: no sporting goods in this store.

In [10]:
form_app([("item", "Product title", "Stainless steel 8-cup pour-over coffee maker", 1)], taxonomy_html, "Classify")

Output()

## Recap and exercises

**Key ideas**
1. **Walk the tree.** Many small `Choice` questions beat one giant list: each decision is easy, and the prompt stays small.
2. **Use the whole distribution.** `.probabilities` lets you beam (keep a close runner-up) and multiply steps into a path probability.
3. **Give the model an exit.** `none_of_these` at every level turns "doesn't fit" into a real answer instead of a confident mistake.
4. **Descriptions are the spec.** Describing a branch by its contents fixed a wrong answer without touching any logic.

**Exercises**
- Set `CLOSE = 0.05` and re-run Step 4. Does the gaming chair still get two paths? Then try `CLOSE = 0.6` and count the extra Jev calls.
- Add a `"Pet Supplies": {"Dogs": [...], "Cats": [...]}` department and re-run: where does the dog food go now?
- Add a rule in code: if the best path probability is below 0.5, send the item to a human review queue.
- Add a `"Unisex"` category under *Clothing* and check whether the t-shirt stops ending on `none_of_these`.